<a href="https://colab.research.google.com/github/PattiAlves/Modelagem_IA/blob/main/1.%20TRATAMENTO%20E%20AN%C3%81LISE%20DE%20DADOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. TRATAMENTO E ANÁLISE DE DADOS**

### **1.1 Conversão de arquivos de precipitação .txt em planilha de dados**

In [ ]:
# 1. Montar o Google Drive (execute apenas se estiver rodando no Google Colab)
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import glob
import os
import pandas as pd

# 1. Montar o Google Drive (ignora se já estiver montado)
try:
  from google.colab import drive

  drive.mount('/content/drive')
except:
  pass

# 2. Caminho da pasta no Drive
pasta_txt = '/content/drive/MyDrive/PUC/EQM2118/CHUVA_2010_2026'

lista_dfs = []
arquivos_txt = glob.glob(os.path.join(pasta_txt, '*.txt'))

# Período estrito da Etapa 1
data_inicio = pd.to_datetime('2010-07-14 17:40:00')
data_fim = pd.to_datetime('2024-10-31 23:59:59')

if not arquivos_txt:
  print('Nenhum arquivo .txt encontrado na pasta especificada.')
else:
  for arquivo in arquivos_txt:
    try:
      # Leitura por largura fixa (fwf) baseada estritamente no layout visual do TXT:
      # 01 a 10: Dia
      # 12 a 20: Hora
      # 35 a 41: Coluna de 01 h (precip_1h)
      colspecs = [(0, 10), (12, 20), (35, 41)]
      names = ['Dia', 'Hora', 'precip_1h']

      df = pd.read_fwf(
          arquivo,
          skiprows=5,
          colspecs=colspecs,
          names=names,
          encoding='latin1',
      )

      df = df.dropna(how='all')
      if df.empty:
        continue

      # Filtra apenas linhas onde a coluna 'Dia' começa com o formato de data (DD/MM/AAAA)
      df = df[df['Dia'].astype(str).str.contains(r'^\d{2}/\d{2}/\d{4}', na=False)]
      if df.empty:
        continue

      # Criação da coluna datetime combinando Dia e Hora
      df['datetime'] = pd.to_datetime(
          df['Dia'].astype(str) + ' ' + df['Hora'].astype(str),
          format='%d/%m/%Y %H:%M:%S',
          errors='coerce',
      )

      # Converte precip_1h para numérico
      df['precip_1h'] = pd.to_numeric(df['precip_1h'], errors='coerce')

      # Seleciona estritamente as colunas desejadas
      df_final = df[['datetime', 'precip_1h']].copy()
      lista_dfs.append(df_final)

    except Exception as e:
      print(f'Erro ao processar o arquivo {os.path.basename(arquivo)}: {e}')

  if lista_dfs:
    # Concatena todos os DataFrames
    df_consolidado = pd.concat(lista_dfs, ignore_index=True)

    # Remove nulos, elimina duplicadas e ordena cronologicamente
    df_consolidado = (
        df_consolidado.dropna(subset=['datetime'])
        .drop_duplicates(subset=['datetime'])
        .sort_values('datetime')
        .reset_index(drop=True)
    )

    # Aplica o filtro temporal estrito da Etapa 1
    df_consolidado = df_consolidado[
        (df_consolidado['datetime'] >= data_inicio)
        & (df_consolidado['datetime'] <= data_fim)
    ]

    # Salva o arquivo final da Etapa 1 no Google Drive
    caminho_saida = os.path.join(
        pasta_txt, 'banco_precipitacoes_etapa1_2010_2024.xlsx'
    )
    df_consolidado.to_excel(caminho_saida, index=False)

    print(
        f'Etapa 1 concluída com sucesso! Arquivo salvo em: {caminho_saida} ('
        f'{len(df_consolidado)} registros)'
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Etapa 1 concluída com sucesso! Arquivo salvo em: /content/drive/MyDrive/PUC/EQM2118/CHUVA_2010_2026/banco_precipitacoes_etapa1_2010_2024.xlsx (499641 registros)


### **1.2 Interpolação de dados de chuva para frequência de 10min**

In [ ]:
import os
import pandas as pd

# 1. Caminho da pasta no Google Drive
pasta_txt = '/content/drive/MyDrive/PUC/EQM2118/CHUVA_2010_2026'
caminho_etapa1 = os.path.join(
    pasta_txt, 'banco_precipitacoes_etapa1_2010_2024.xlsx'
)

# 2. Carrega a base gerada na Etapa 1
df_etapa1 = pd.read_excel(caminho_etapa1)

# Converte datetime para o tipo adequado e define como índice
df_etapa1['datetime'] = pd.to_datetime(df_etapa1['datetime'])
df_etapa1 = df_etapa1.sort_values('datetime').reset_index(drop=True)

# 3. Define a data de início exata solicitada (14/07/2010 18:00:00)
data_inicio = pd.to_datetime('2010-07-14 18:00:00')
df_etapa1 = df_etapa1[df_etapa1['datetime'] >= data_inicio]

# 4. Criação do novo grid temporal de 10 em 10 minutos
data_fim = df_etapa1['datetime'].max()
grid_10min = pd.date_range(start=data_inicio, end=data_fim, freq='10min')

# Cria um DataFrame temporário com o grid de 10 min
df_10min = pd.DataFrame({'datetime': grid_10min})

# 5. Mesclagem e interpolação dos dados para a frequência de 10 minutos
# Como a precipitação acumulada de 1h varia suavemente ou pode ser distribuída uniformemente/linearmente:
df_merged = pd.merge_asof(
    df_10min, df_etapa1, on='datetime', direction='nearest'
)

# Se preferir interpolação linear dos valores de 1h para estimar os passos intermediários de 10 min:
df_merged['precip_1h_interpolada'] = df_merged['precip_1h'].interpolate(
    method='linear'
)

# Para obter a precipitação de 10 min (precip_10min), divide-se a taxa horária proporcionalmente por 6
# ou aplica-se o método de desagregação desejado:
df_merged['precip_10min'] = df_merged['precip_1h_interpolada'] / 6.0

# Seleciona as colunas finais solicitadas: 'datetime', 'precip_10min' e 'precip_1h'
df_final_10min = df_merged[['datetime', 'precip_10min', 'precip_1h_interpolada']].copy()
df_final_10min.rename(columns={'precip_1h_interpolada': 'precip_1h'}, inplace=True)

# 6. Salva a nova base interpolada de 10 minutos no Google Drive
caminho_saida_10min = os.path.join(
    pasta_txt, 'banco_precipitacoes_10min_2010_2024.xlsx'
)
df_final_10min.to_excel(caminho_saida_10min, index=False)

print(f'Etapa 2 concluída com sucesso! Arquivo salvo em: {caminho_saida_10min}')
print(f'Total de registros de 10 min gerados: {len(df_final_10min)}')

Etapa 2 concluída com sucesso! Arquivo salvo em: /content/drive/MyDrive/PUC/EQM2118/CHUVA_2010_2026/banco_precipitacoes_10min_2010_2024.xlsx
Total de registros de 10 min gerados: 752147


In [ ]:
import os
import pandas as pd

# Caminho da base gerada na Etapa 2
pasta_txt = '/content/drive/MyDrive/PUC/EQM2118/CHUVA_2010_2026'
caminho_10min = os.path.join(
    pasta_txt, 'banco_precipitacoes_10min_2010_2024.xlsx'
)

# Carrega a base de 10 minutos
df = pd.read_excel(caminho_10min)
df['datetime'] = pd.to_datetime(df['datetime'])

print('--- 1. ESTATÍSTICAS BÁSICAS ---')
print(df[['precip_10min', 'precip_1h']].describe())

print('\n--- 2. VERIFICAÇÃO DE VALORES NEGATIVOS ---')
negativos = (df['precip_10min'] < 0).sum()
print(f'Quantidade de valores negativos encontrados: {negativos}')

print('\n--- 3. TESTE DE CONSERVAÇÃO DE VOLUME (POR HORA) ---')
# Agrupa os valores de 10 min por hora e compara com a média/soma esperada
df['hora_truncada'] = df['datetime'].dt.floor('h')
comparacao = (
    df.groupby('hora_truncada')
    .agg({'precip_10min': 'sum', 'precip_1h': 'mean'})
    .reset_index()
)

print(comparacao.head(10))

--- 1. ESTATÍSTICAS BÁSICAS ---
        precip_10min      precip_1h
count  752147.000000  752147.000000
mean        0.019723       0.118335
std         0.173295       1.039770
min         0.000000       0.000000
25%         0.000000       0.000000
50%         0.000000       0.000000
75%         0.000000       0.000000
max        18.266667     109.600000

--- 2. VERIFICAÇÃO DE VALORES NEGATIVOS ---
Quantidade de valores negativos encontrados: 0

--- 3. TESTE DE CONSERVAÇÃO DE VOLUME (POR HORA) ---
        hora_truncada  precip_10min  precip_1h
0 2010-07-14 18:00:00      0.766667   0.766667
1 2010-07-14 19:00:00      0.666667   0.666667
2 2010-07-14 20:00:00      1.866667   1.866667
3 2010-07-14 21:00:00      0.900000   0.900000
4 2010-07-14 22:00:00      0.300000   0.300000
5 2010-07-14 23:00:00      0.733333   0.733333
6 2010-07-15 00:00:00      0.100000   0.100000
7 2010-07-15 01:00:00      0.000000   0.000000
8 2010-07-15 02:00:00      0.000000   0.000000
9 2010-07-15 03:00:00      0

### **1.3 Consolidação dados de chuva de 2010 a 2026**

In [ ]:
import glob
import os
import pandas as pd

# 1. Montar o Google Drive (ignora se já estiver montado)
try:
  from google.colab import drive

  drive.mount('/content/drive')
except:
  pass

# 2. Caminho da pasta no Drive
pasta_txt = '/content/drive/MyDrive/PUC/EQM2118/CHUVA_2010_2026'

lista_dfs_novos = []
arquivos_txt = glob.glob(os.path.join(pasta_txt, '*.txt'))

# Período dos dados com frequência real de 10 minutos
data_inicio_etapa2 = pd.to_datetime('2024-11-01 00:00:00')
data_fim_etapa2 = pd.to_datetime('2026-09-19 23:59:59')

if not arquivos_txt:
  print('Nenhum arquivo .txt encontrado na pasta especificada.')
else:
  for arquivo in arquivos_txt:
    try:
      with open(arquivo, 'r', encoding='latin1') as f:
        linhas = f.readlines()

      header_tokens = None
      for l in linhas:
        if 'Dia' in l and 'Hora' in l:
          raw_tokens = l.strip().split()
          header_tokens = []
          i = 0
          while i < len(raw_tokens):
            t = raw_tokens[i]
            if (
                t in ['05', '10', '15']
                and i + 1 < len(raw_tokens)
                and raw_tokens[i + 1] == 'min'
            ):
              header_tokens.append(f'{t} min')
              i += 2
            elif (
                t in ['01', '04', '24', '96']
                and i + 1 < len(raw_tokens)
                and raw_tokens[i + 1] == 'h'
            ):
              header_tokens.append(f'{t} h')
              i += 2
            else:
              header_tokens.append(t)
              i += 1
          break

      if not header_tokens:
        continue

      dados_arquivo = []
      for linha in linhas:
        parts = linha.strip().split()
        if len(parts) >= 2 and '/' in parts[0] and ':' in parts[1]:
          dia = parts[0]
          hora = parts[1]
          has_hbv = len(parts) > 2 and parts[2] == 'HBV'
          start_idx = 3 if has_hbv else 2
          data_tokens = parts[start_idx:]

          row_dict = {'Dia': dia, 'Hora': hora}
          for idx, col in enumerate(header_tokens[2:]):
            if idx < len(data_tokens):
              row_dict[col] = data_tokens[idx]
            else:
              row_dict[col] = None
          dados_arquivo.append(row_dict)

      if not dados_arquivo:
        continue

      df = pd.DataFrame(dados_arquivo)
      df['datetime'] = pd.to_datetime(
          df['Dia'].astype(str) + ' ' + df['Hora'].astype(str),
          format='%d/%m/%Y %H:%M:%S',
          errors='coerce',
      )

      # Filtra estritamente o período de novembro de 2024 em diante
      df = df[
          (df['datetime'] >= data_inicio_etapa2)
          & (df['datetime'] <= data_fim_etapa2)
      ]
      if df.empty:
        continue

      # Extrai precip_10min real e precip_1h
      if '10 min' in df.columns:
        df['precip_10min'] = pd.to_numeric(df['10 min'], errors='coerce')
      else:
        df['precip_10min'] = pd.NA

      if '01 h' in df.columns:
        df['precip_1h'] = pd.to_numeric(df['01 h'], errors='coerce')
      else:
        continue

      df_final = df[['datetime', 'precip_10min', 'precip_1h']].copy()
      lista_dfs_novos.append(df_final)

    except Exception as e:
      print(f'Erro ao processar o arquivo {os.path.basename(arquivo)}: {e}')

  if lista_dfs_novos:
    df_etapa2 = pd.concat(lista_dfs_novos, ignore_index=True)

    # Carrega a base anterior de 10 minutos (2010–2024)
    caminho_10min_antigo = os.path.join(
        pasta_txt, 'banco_precipitacoes_10min_2010_2024.xlsx'
    )

    if os.path.exists(caminho_10min_antigo):
      df_base_antiga = pd.read_excel(caminho_10min_antigo)
      df_base_antiga['datetime'] = pd.to_datetime(df_base_antiga['datetime'])

      # Concatena a base interpolada anterior com a nova base de dados reais de 10 min
      df_completo = pd.concat(
          [df_base_antiga, df_etapa2], ignore_index=True
      )

      # Limpeza final, remoção de duplicadas e ordenação
      df_completo = (
          df_completo.dropna(subset=['datetime'])
          .drop_duplicates(subset=['datetime'])
          .sort_values('datetime')
          .reset_index(drop=True)
      )

      # Salva o arquivo unificado final
      caminho_saida_final = os.path.join(
          pasta_txt, 'banco_precipitacoes_completo_2010_2026.xlsx'
      )
      df_completo.to_excel(caminho_saida_final, index=False)

      print(
          'Processo concluído com sucesso! Base completa unificada salva em:'
          f' {caminho_saida_final} ({len(df_completo)} registros totais)'
      )
    else:
      print(
          'O arquivo da base anterior (Etapa 2) não foi encontrado. Certifique-se'
          ' de executá-lo primeiro.'
      )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processo concluído com sucesso! Base completa unificada salva em: /content/drive/MyDrive/PUC/EQM2118/CHUVA_2010_2026/banco_precipitacoes_completo_2010_2026.xlsx (949102 registros totais)


### **1.4 Conversão de arquivo de maré .txt em planilha de dados**

In [ ]:
import glob
import os
import pandas as pd

# 1. Montar o Google Drive
try:
  from google.colab import drive

  drive.mount('/content/drive')
except:
  pass

# 2. Defina o caminho da pasta no Drive onde estão os TXT de maré
# (Ajuste a subpasta caso os txts estejam em outro diretório dentro de EQM2118)
pasta_txt_mare = '/content/drive/MyDrive/PUC/EQM2118/MARÉ_2010_2026'

lista_dfs_mare = []
arquivos_txt = glob.glob(os.path.join(pasta_txt_mare, '*.txt'))

if not arquivos_txt:
  print('Nenhum arquivo .txt de maré encontrado na pasta especificada.')
else:
  for arquivo in arquivos_txt:
    try:
      # Lendo o arquivo txt (ajuste o skiprows se houver cabeçalho descritivo similar aos de chuva)
      # Se os dados começarem direto, pode remover o skiprows ou ajustá-lo conforme o layout do seu txt de maré
      df = pd.read_csv(
          arquivo,
          skiprows=4,
          sep=r'\s+',
          header=None,
          engine='python',
          encoding='latin1',
      )

      df = df.dropna(how='all')
      if df.empty:
        continue

      # Considerando que o layout contenha 'Dia' na coluna 0 e 'Hora' na coluna 1, e o nível de maré na coluna subsequente
      # Filtra linhas para garantir que a coluna 'Dia' comece com o formato de data (ex: '01/...')
      df = df[df.iloc[:, 0].str.contains(r'^\d{2}/\d{2}/\d{4}', na=False)]

      if df.shape[1] >= 3:
        # Seleciona Dia (col 0), Hora (col 1) e Nível (col 2 ou ajuste conforme a posição real no seu txt de maré)
        df_temp = df.iloc[:, [0, 1, 2]].copy()
        df_temp.columns = ['Dia', 'Hora', 'nivel']

        # Criando a coluna datetime combinando Dia e Hora
        df_temp['datetime'] = pd.to_datetime(
            df_temp['Dia'].astype(str) + ' ' + df_temp['Hora'].astype(str),
            format='%d/%m/%Y %H:%M:%S',
            errors='coerce',
        )

        # Trata o nível substituindo vírgula por ponto se vier como texto e converte para float
        df_temp['nivel'] = (
            df_temp['nivel']
            .astype(str)
            .str.replace(',', '.')
            .astype(float, errors='ignore')
        )

        # Mantém apenas as colunas solicitadas
        df_final = df_temp[['datetime', 'nivel']].dropna(subset=['datetime'])
        lista_dfs_mare.append(df_final)

    except Exception as e:
      print(f'Erro ao processar o arquivo de maré {os.path.basename(arquivo)}: {e}')

  if lista_dfs_mare:
    # 1. Concatena todos os arquivos de maré
    df_mare_consolidado = pd.concat(lista_dfs_mare, ignore_index=True)

    # 2. Remove duplicadas e ordena cronologicamente
    df_mare_consolidado = (
        df_mare_consolidado.drop_duplicates(subset=['datetime'])
        .sort_values('datetime')
        .reset_index(drop=True)
    )

    # Formata a coluna datetime para o padrão de string DD/MM/YYYY HH:MM conforme esperado pelo script de spline
    df_mare_consolidado['datetime'] = df_mare_consolidado['datetime'].dt.strftime(
        '%d/%m/%Y %H:%M'
    )

    # 3. Salva com o nome exato solicitado na mesma pasta do Google Drive
    caminho_saida_excel = os.path.join(pasta_txt_mare, 'Maré_2010_2024.xlsx')
    df_mare_consolidado.to_excel(
        caminho_saida_excel, index=False
    )  # Sem header se o seu script lê header=None e skiprows=1

    print(
        'Planilha de maré gerada e salva com sucesso em:'
        f' {caminho_saida_excel} ({len(df_mare_consolidado)} registros)'
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Erro ao processar o arquivo de maré 50174_01012006_20092026_PREV_MAX_MIN.txt: Expected 2 fields in line 6, saw 6. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.


### **1.5 Interpolação Spline Cubica dos dados de maré para frequência de 10min**

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline

# 1. Montar o Google Drive
from google.colab import drive

drive.mount('/content/drive')

# Definir o caminho da pasta no Drive
pasta_mare = '/content/drive/MyDrive/PUC/EQM2118/MARÉ_2010_2026'
caminho_arquivo = os.path.join(pasta_mare, 'Maré_2010_2024.xlsx')

# 2. Carregar os dados de maré de baixa frequência do arquivo Excel no Drive
# (Adicionado skiprows=1 para pular a primeira linha com os rótulos de coluna)
df_maré = pd.read_excel(caminho_arquivo, header=None, skiprows=1)

# Atribuir nomes corretos às colunas e converter tipos
df_maré.columns = ['datetime', 'nivel']
df_maré['datetime'] = pd.to_datetime(df_maré['datetime'], format='%d/%m/%Y %H:%M')
df_maré['nivel'] = (
    df_maré['nivel'].astype(str).str.replace(',', '.').astype(float)
)

# Garantir que os dados estejam ordenados cronologicamente
df_maré = df_maré.sort_values('datetime').reset_index(drop=True)

# 3. Converter o índice temporal em segundos (valores numéricos) para a interpolação matemática
tempo_num = df_maré['datetime'].astype(int) / 10**9
niveis = df_maré['nivel'].values

# 4. Criar a função de Interpolação por Spline Cúbica
cs = CubicSpline(tempo_num, niveis, bc_type='not-a-knot')

# 5. Definir a nova grade temporal de alta resolução (a cada 10 minutos)
# Forçando o início exatamente na data solicitada
data_inicio = pd.to_datetime('2010-07-14 17:00:00')
data_fim = (
    df_maré['datetime'].max()
)  # Vai até a última data disponível na sua planilha

nova_grade = pd.date_range(start=data_inicio, end=data_fim, freq='10min')

# 6. Converter a nova grade para o formato numérico e aplicar a spline
tempo_novo_num = nova_grade.astype(int) / 10**9
niveis_interpolados = cs(tempo_novo_num)

# 7. Montar o DataFrame final com a maré interpolada a cada 10 minutos
df_maré_10min = pd.DataFrame(
    {'datetime': nova_grade, 'nivel_mare_10min': niveis_interpolados}
)

# Visualizar as primeiras linhas do resultado
print(df_maré_10min.head())

# 8. Salvar o resultado de volta na mesma pasta do Google Drive
caminho_saida = os.path.join(pasta_mare, 'maré_10min.xlsx')
df_maré_10min.to_excel(caminho_saida, index=False)
print(f'Arquivo de maré interpolado salvo com sucesso em: {caminho_saida}')

## **2. TREINAMENTO DE MODELO E TESTE**

## **3. RECONSTRUÇÃO SÉRIE DE DADOS DE NÍVEL DA LAGOA**

## **4. CORRELAÇÃO DE SPEARMAN DE DADOS DA LAGOA COM AS COTAS DE INUNDAÇÃO**